# Week 3 Day 3 (free route) - The context layer and memory

## Running UnderCover's context layer on your laptop

On the AWS route, today's lab puts the context layer on Lambda as its own MCP server, behind the AgentCore Gateway with the app, and gives the agent an AgentCore Memory. On the free route the context layer runs on your laptop beside the agent, the agent calls it and the app directly, and the agent works without a memory.

The Gateway and the Memory are AWS services, so for those you can watch me. Everything else in today's build works the same way on your laptop: the agent looks things up in the context layer, each lookup says where it came from, and the decision record holds its calculation.

This notebook has no code to run. It gives you the steps.

## 4 steps:

STEP 1: Update the app on Render

STEP 2: Look at the context layer

STEP 3: Run a returning customer's claims

STEP 4: Compare with the AWS route

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of today's lab. The main lab, <a href="week3day3.ipynb">week3day3.ipynb</a>, runs on AWS. You need the Render service from Day 2, and today's build of the product, yours or <code>week3day3-end</code>.<br/><br/>Step 3 works two claims, which uses part of your 50 free requests for the day. If you run out, the requests come back the next day.
            </span>
        </td>
    </tr>
</table>

## Step 1: Update the app on Render

Today's build changes the app: the documents moved to the context layer, and the decision record checks itself against the claims manual. So the app on Render needs today's code.

- **To deploy my build,** open your web service in the [Render Dashboard](https://dashboard.render.com), go to **Settings**, change the **Branch** to `week3day3-end`, and save. Then choose **Manual Deploy** and **Deploy latest commit**.
- **To deploy your own build,** push your branch to your own GitHub repository, as on Day 2, and choose **Manual Deploy** and **Deploy latest commit**.

The build and start commands and the environment variables stay as they were on Day 2. The context layer doesn't go on Render: it runs on your laptop beside the agent, which is the only thing that calls it.

Render's documentation: [Deploys](https://render.com/docs/deploys).

## Step 2: Look at the context layer

Open the `context` folder in the `undercover` repository. It holds seven files, each standing in for another system at the firm or an outside provider, and each naming its source system, its owner and when it was last reviewed:

| File | What it holds | Where it would come from |
|---|---|---|
| `data_dictionary.yaml` | The meaning of each field the agent sees | The data catalog |
| `glossary.yaml` | The terms the policy wording defines | The policy wording |
| `plans.yaml` | Each plan's limits, waiting periods and deductible | Policy administration |
| `claim_rules.yaml` | The filing window, the look-back, repeat claims and meals | Claims operations |
| `reference_data.yaml` | Outside events, places and products | Outside data providers |
| `policy_wording.md` | The contract with the customer, by section | Legal |
| `claims_manual.md` | How handlers work a claim, by section | Claims operations |

`npm run dev` now starts the context layer's MCP server beside the backend, the agent's service and the dashboard, at `http://127.0.0.1:8006/mcp`. When there's no Gateway in `.env`, the agent calls it there directly, and calls the app directly with its API key, as it did before today.

## Step 3: Run a returning customer's claims

Oliver Grant (CUS-3005) has two claims in the queue. In July a giraffe ate his passport in Nairobi, and in August a coconut fell on him in Sri Lanka. His second claim opens "Yes, it is me again".

In a terminal in the `undercover` folder, reseed so both claims are waiting, then start everything:

```
npm run seed
npm run dev
```

In a second terminal, work the two claims, one after the other:

```
uv run -m agent CLM-1005
uv run -m agent CLM-1006
```

Open each claim's page on the dashboard, at `http://localhost:5173/claims/CLM-1005` and `http://localhost:5173/claims/CLM-1006`, or on your app on Render, signed in as Dana Whitfield. Both read the same Supabase database. Look for:

- the lookups in the context layer, each saying which file it came from and who owns it;
- the Context used panel, with everything the agent looked up, grouped by file;
- the decision record, with the calculation line by line, who made the decision and the claims manual's version.

There is no memory step on CLM-1006. Without a memory, the agent knows only what our records say about Oliver, and his claim history in the records is empty.

## Step 4: Compare with the AWS route

In the main lab, the same two claims run with the Gateway and the Memory. Watch Part 5 there: once the agent has decided CLM-1005, AgentCore turns the claim and the decision into facts about Oliver and a summary of the claim, and on CLM-1006 the agent can search them and find the giraffe.

There's nothing to tear down. Your app on Render costs nothing, so keep it for Day 4.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: give the agent a memory of its own without AWS. LangGraph's <code>PostgresStore</code> can keep it in your Supabase database, with pgvector to search it, in a namespace for each customer like the AWS route's. Save the claim and the decision when the agent has decided, and give the agent a tool to search its customer's memories.<br/><br/>I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>